# Grouped Splits and Leakage Audit

**Question:** How do we prevent dependent pairs from crossing splits?

**Prerequisites:** install the project with notebook extras; run from the repository root. This notebook is self-contained in bounded demo mode.

## Theory and assumptions

Statistical independence belongs to an acquisition or patient, not a patch. One geometry can produce many scatterer realizations, compression levels and frame pairs. For a geometry-generalization question, all descendants remain in one partition. If true group metadata is absent, duplicate detection does not prove independence. Separate train, validation, calibration and test roles protect fitting, model selection, uncertainty tuning and final evaluation.

In [ ]:
from pathlib import Path
import os,json
import numpy as np
import matplotlib.pyplot as plt
ROOT=Path.cwd()
if not (ROOT/'configs').exists():ROOT=ROOT.parent
mode='demo'
data_root=ROOT/'data/raw'
config_path=ROOT/'configs/demo.yaml'
seed=42
max_samples=8
device='cpu'
output_dir=ROOT/'reports/notebooks'/'04_Grouped_Splits_and_Leakage_Audit'
output_dir.mkdir(parents=True,exist_ok=True)
print('Mode:',mode,'| Data origin: synthetic_fixture')


## Experiment

Inputs: seeded synthetic fixtures or explicitly identified educational probability tables. Outputs: printed numerical evidence, saved run artifacts where applicable, and plots. Real-data-only claims remain unavailable until source archives and metadata are verified.

In [ ]:
from elastograde.data.splits import grouped_split
from elastograde.data.audit import audit
records=[{'sample_id':f's{i}','geometry_id':f'g{i//2}','frame_ids':[f'f{i}']} for i in range(24)]
split=grouped_split(records,seed=seed)
print(audit(split));print({role:sum(r['split']==role for r in split) for role in ['train','validation','calibration','test']})
a=split[0];b=next(r for r in split if r['split']!=a['split']);b['frame_ids']=a['frame_ids']
try:audit(split)
except ValueError as e:print('Expected rejection:',e)

## Interpretation and failure cases

The deliberate shared-frame leak is rejected. This protects declared lineage; it cannot recover unrecorded acquisition identity.

## Check your understanding

Why split before patch generation? Answer: otherwise nearly identical patches from the same source can appear in training and testing.

## Evidence statement

This execution demonstrates the computations shown above. It does not establish clinical accuracy, generalization to patients, or state-of-the-art performance. Record the environment and split protocol before a full experiment.